In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms

# NEW DEPRENDENCY: Make sure to pip install timm
import timm



In [2]:

# 1. Custom Dataset to load and label /images dynamically
class LocalImageDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_dir = image_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []

        # Scan directory and label based on file extension
        if os.path.exists(image_dir):
            for filename in os.listdir(image_dir):

                if len(self.image_paths) >= 5000:
                    break

                filepath = os.path.join(image_dir, filename)
                ext = filename.lower()

                if ext.endswith('.png'):
                    self.image_paths.append(filepath)
                    self.labels.append(0)  # PNG = 0
                elif ext.endswith('.jpg') or ext.endswith('.jpeg'):
                    self.image_paths.append(filepath)
                    self.labels.append(1)  # JPG/JPEG = 1
        else:
            print(f"Warning: Directory {image_dir} not found.")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        # Channel consistency (handles grayscale, RGB, or 4-channel RGBA)
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, label



In [3]:
# 2. Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 3. Data transforms tailored for ViT (ViT needs explicit 224x224 patch matching)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    # Standard ImageNet normalization used by pretrained ViT
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset from /images
dataset = LocalImageDataset(image_dir='E:\images', transform=transform)

if len(dataset) == 0:
    raise ValueError("No matching .jpg or .png images found in /images directory.")

train_loader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=0)



Using device: cuda


In [4]:
# 4. Load Tiny Vision Transformer (ViT Tiny) and modify output layer
# This creates a ~5.7M parameter transformer model with pretrained weights
model = timm.create_model('vit_tiny_patch16_224', pretrained=True)

# Replace the classification head for 2 classes (PNG vs JPG)
# Note: ViT heads are named 'head' in timm, instead of 'fc' like in ResNet
num_ftrs = model.head.in_features
model.head = nn.Linear(num_ftrs, 2)
model = model.to(device)

# 5. Loss function and optimizer (AdamW is strongly preferred for transformers over SGD)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.05)



In [5]:
# 6. Training loop
num_epochs = 10
model.train()

for epoch in range(num_epochs):
    running_loss = 0.0
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if (i + 1) % max(1, len(train_loader) // 5) == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {running_loss/max(1, i+1):.4f}")

print("Finished Training")



Epoch [1/10], Step [62/313], Loss: 1.0406
Epoch [1/10], Step [124/313], Loss: 0.8763
Epoch [1/10], Step [186/313], Loss: 0.8088
Epoch [1/10], Step [248/313], Loss: 0.7729
Epoch [1/10], Step [310/313], Loss: 0.7513
Epoch [2/10], Step [62/313], Loss: 0.6487
Epoch [2/10], Step [124/313], Loss: 0.6801
Epoch [2/10], Step [186/313], Loss: 0.6757
Epoch [2/10], Step [248/313], Loss: 0.6746
Epoch [2/10], Step [310/313], Loss: 0.6714
Epoch [3/10], Step [62/313], Loss: 0.6611
Epoch [3/10], Step [124/313], Loss: 0.6505
Epoch [3/10], Step [186/313], Loss: 0.6545
Epoch [3/10], Step [248/313], Loss: 0.6515
Epoch [3/10], Step [310/313], Loss: 0.6446
Epoch [4/10], Step [62/313], Loss: 0.1593
Epoch [4/10], Step [124/313], Loss: 0.0797
Epoch [4/10], Step [186/313], Loss: 0.0531
Epoch [4/10], Step [248/313], Loss: 0.0398
Epoch [4/10], Step [310/313], Loss: 0.0319
Epoch [5/10], Step [62/313], Loss: 0.0000
Epoch [5/10], Step [124/313], Loss: 0.0000
Epoch [5/10], Step [186/313], Loss: 0.0000
Epoch [5/10], St

# 7. Save model parameters as a .pth file
model_save_path = "./vit_tiny_image_classifier.pth"
os.makedirs(os.path.dirname(model_save_path), exist_ok=True)
torch.save(model.state_dict(), model_save_path)
print(f"Model saved successfully to {model_save_path}")

In [6]:
# 7. COMPILE AND SAVE WITH TORCHSCRIPT (Self-contained file)
model.eval()  # Must place in evaluation mode before tracing
with torch.inference_mode():
    dummy_input = torch.rand(1, 3, 224, 224).to(device)
    # Trace compiles the model structure + parameter weights into an intermediate graph
    traced_model = torch.jit.trace(model, dummy_input)

    model_save_path = "./vit_tiny_compiled.pt"
    os.makedirs(os.path.dirname(model_save_path), exist_ok=True)
    traced_model.save(model_save_path)
    print(f"Tiny ViT serialized successfully to {model_save_path}")

C:\Users\starfuryz2040\DataspellProjects\csephase2cnn\venv\lib\site-packages\torch\__init__.py:2132: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert condition, message


Tiny ViT serialized successfully to ./vit_tiny_compiled.pt
